# Imports

In [2]:
from roboflow import Roboflow
import os
from pathlib import Path
from ultralytics import YOLO
import shutil
import cv2
import numpy as np

In [3]:
dataset_path = Path(".")
current_dir = Path.cwd()

# Dataset Loading

In [3]:
load_data = False

In [4]:
if load_data:
    os.chdir(dataset_path)

    rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
    workspace_name = "arena-qye1f"
    project_name = "billiard-table-keypoint-lilj3"
    project = rf.workspace(workspace_name).project(project_name)
    versions = project.versions()

    latest_version = max(versions, key=lambda v: int(v.version))

    print(f"  -> Downloading version {latest_version.version}...")
    latest_version.download("yolov8")  # Download in YOLOv8 format

    os.chdir(current_dir)

# Train

In [ ]:
model = YOLO("yolov8n-pose.pt")


results = model.train(
    data="data/Billiard-Table-Keypoint-5/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device="0",
    workers=0,
    name="pool_table_model",
    project="runs/pose",
    exist_ok=True,
)

best_model_path = results.save_dir / "weights" / "best.pt"
final_path = "pool_table_detector_best.pt"

shutil.copy(best_model_path, final_path)

print(f"Best model saved at: {best_model_path}")
print(f"Copied final model to: {final_path}")

metrics = YOLO(best_model_path).val(
    data="data/Billiard-Table-Keypoint-5/data.yaml",
    workers=0,
    batch=8,
    device="0",
)

print(f"mAP50: {metrics.pose.map50:.4f}")
print(f"mAP50-95: {metrics.pose.map:.4f}")

New https://pypi.org/project/ultralytics/8.4.79 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.67 🚀 Python-3.12.9 torch-2.6.0+cu124 CUDA:0 (Quadro RTX 8000, 48395MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/Billiard-Table-Keypoint-5/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-pose.pt, momentum=0.937, mosaic=1.0, multi_

TypeError: unsupported format string passed to dict.__format__

# Test on main_dataset

In [6]:
model = YOLO("pool_table_detector_best.pt")

## Perform homography

In [19]:
def homography(img, kpts):

    src_pts = kpts.astype(np.float32)

    # IMPORTANT: must match correct ordering!
    # adjust if needed based on dataset consistency
    dst_pts = np.array([
        [0, 0],
        [640, 0],
        [640, 480],
        [0, 480]
    ], dtype=np.float32)

    H = cv2.getPerspectiveTransform(src_pts, dst_pts)

    warped_img = cv2.warpPerspective(img, H, (640, 480))

    return warped_img

## Apply model to images

In [ ]:
import os
import cv2
import numpy as np

target_images = "../datasets/main_dataset/data/train"

output_dir = "./results/run1"
os.makedirs(output_dir, exist_ok=True)

count = 0

for image_file in sorted(os.listdir(target_images)):

    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(target_images, image_file)
    img = cv2.imread(img_path)

    results = model.predict(
        img_path,
        save=False,
        verbose=False
    )

    result = results[0]

    if result.keypoints is None or len(result.keypoints.xy) == 0:
        continue

    plotted = result.plot()
    cv2.imwrite(os.path.join(output_dir, f"pred_{image_file}"), plotted)

    kpts = result.keypoints.xy.cpu().numpy()[0].astype(np.float32)
    warped = homography(img, kpts)
    cv2.imwrite(os.path.join(output_dir, f"homography_{image_file}"), warped)

NameError: name 'model' is not defined

In [5]:
metrics = YOLO("pool_table_seg_best.pt").val(
    data="pool-table-segmentation-1/data.yaml",
    device="0",
    batch=8,
    workers=0,
)
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50 (seg): {metrics.seg.map50:.4f}")
print(f"mAP50-95 (seg): {metrics.seg.map:.4f}")

# %% [markdown]
# # Test on main_dataset

# %%
model = YOLO("pool_table_seg_best.pt")

# %% [markdown]
# ## Perform homography

# %%
def homography(img, kpts):

    src_pts = kpts.astype(np.float32)

    # IMPORTANT: must match correct ordering!
    # adjust if needed based on dataset consistency
    dst_pts = np.array([
        [0, 0],
        [640, 0],
        [640, 480],
        [0, 480]
    ], dtype=np.float32)

    H = cv2.getPerspectiveTransform(src_pts, dst_pts)

    warped_img = cv2.warpPerspective(img, H, (640, 480))

    return warped_img

# %% [markdown]
# ## Apply model to images

# %%
def get_corners_from_polygon(polygon):
    # polygon: (N,2)

    hull = cv2.convexHull(polygon.astype(np.float32))

    epsilon = 0.02 * cv2.arcLength(hull, True)
    approx = cv2.approxPolyDP(hull, epsilon, True)

    # If not exactly 4 points, force fallback
    if len(approx) != 4:
        rect = cv2.minAreaRect(hull)
        box = cv2.boxPoints(rect)
        return box.astype(np.float32)

    return approx.reshape(4, 2).astype(np.float32)

# %%
def get_corners_fast(polygon):
    rect = cv2.minAreaRect(polygon.astype(np.float32))
    box = cv2.boxPoints(rect)
    return box.astype(np.float32)

# %%
target_images = "../datasets/main_dataset/data/train"
output_dir = "./results/run1"
os.makedirs(output_dir, exist_ok=True)

count = 0

for image_file in sorted(os.listdir(target_images)):

    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(target_images, image_file)
    img = cv2.imread(img_path)

    results = model.predict(img_path, verbose=False)
    result = results[0]

    if result.masks is None:
        print(f"No table detected: {image_file}")
        continue

    polygon = result.masks.xy[0]

    kpts = get_corners_from_polygon(polygon)

    save_img = homography(img, kpts)

    cv2.imwrite(
        os.path.join(output_dir, f"homography_{image_file}"),
        save_img
    )

    count += 1

print(f"Processed {count} images.")




Ultralytics 8.4.67 🚀 Python-3.12.9 torch-2.6.0+cu124 CUDA:0 (Quadro RTX 8000, 48395MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2126.5±477.1 MB/s, size: 52.9 KB)
val: Scanning /home/admin/Documents/FEUP-CV/proj/task3.2/pool-table-segmentation-1/valid/labels.cache... 14 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14/14 4.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.0it/s 2.0s3.4s
                   all         14         14      0.996          1      0.995      0.995      0.996          1      0.995      0.995
Speed: 2.0ms preprocess, 8.7ms inference, 0.0ms loss, 22.1ms postprocess per image
Results saved to /home/admin/Documents/FEUP-CV/runs/segment/val-3
mAP50: 0.9950
mAP50-95: 0.9950
mAP50 (seg): 0.9950
mAP50-95 (seg): 0.9950
No table detected: 0_png.rf.e350